Reloading the sample

In [29]:
import pandas as pd
df = pd.read_csv("../data/raw/loan.csv", low_memory=False)
df = df.sample(n=100000, random_state=42)

Select initial candidate columns 

In [30]:
candidate_cols = [
    "loan_amnt", "int_rate", "annual_inc", "emp_length", "grade", "home_ownership",
    "purpose", "loan_status", "issue_d", "dti"]
df = df[candidate_cols]

In [31]:
df["int_rate"].dtype

dtype('float64')

Clean the interest rate column by removing the '%' sign

In [32]:
df["int_rate"] = (
    df["int_rate"]
      .astype(str)
      .str.replace("%", "", regex=False)
      .astype(float)
)

Clean the employment length column by removing "year/s" and "+", treating <1 year as 0, "n/a" as None

In [33]:
df["emp_length"] = (
    df["emp_length"]
      .str.replace("years", "", regex=False)
      .str.replace("year", "", regex=False)
      .str.replace("+", "", regex=False)
      .str.strip()
)

df["emp_length"] = df["emp_length"].replace({
    "< 1": "0",
    "n/a": None
})


Normalize loan status column to contain only completed loans and label them as "non_default" for "Fully Paid", and "default" for "Charged Off"

In [34]:
status_map = {
    "Fully Paid": "non_default",
    "Charged Off": "default"
}

df = df[df["loan_status"].isin(status_map.keys())]
df["loan_status"] = df["loan_status"].map(status_map)


Handling missing values in annual income and employment length columns by replacing:<br> 
> missing values in annual income by the median annual income
> missing values in employment length with Unknown

In [35]:
df["annual_inc"] = df["annual_inc"].fillna(df["annual_inc"].median())
df["emp_length"] = df["emp_length"].fillna("Unknown")

In [37]:
df['issue_d'].head()

1758049   2013-06-01
900721    2016-10-01
1727912   2013-09-01
2153213   2017-12-01
1360732   2017-03-01
Name: issue_d, dtype: datetime64[ns]

Parsing issue_d column into pandas datetime format

In [38]:
df["issue_d"] = pd.to_datetime(df["issue_d"], format="%Y-%m-%d")

Cleaning the dti column to remove negative values identified

In [41]:
df["dti"].dtype

dtype('float64')

In [42]:
df["dti"].isna().mean()

np.float64(0.00024322023592362885)

In [44]:
df["dti"].describe()

count    57547.00000
mean        18.23952
std          8.94394
min         -1.00000
25%         11.78000
50%         17.68000
75%         24.07000
max        266.77000
Name: dti, dtype: float64

Removing negative values from dti

In [47]:
df.loc[df["dti"] < 0, "dti"] = pd.NA

df["dti"].describe()

count    57546.000000
mean        18.239854
std          8.943658
min          0.000000
25%         11.780000
50%         17.680000
75%         24.070000
max        266.770000
Name: dti, dtype: float64

Also, checking how many dti's are >100

In [50]:
(df["dti"] > 100).sum()

np.int64(11)

In [52]:
(df["dti"] > 100).mean()

np.float64(0.00019110161393999408)

In [61]:
df["dti"].isna().sum()

np.int64(15)

The proportion if large outliers and nulls is not high, so not removing them

Finally, removing the duplicates

In [62]:
df = df.drop_duplicates()

In [63]:
import os

os.makedirs("../data/processed", exist_ok=True)

df.to_csv("../data/processed/clean_lending_club.csv", index=False)

print("Cleaned dataset saved successfully.")
print("Shape:", df.shape)

Cleaned dataset saved successfully.
Shape: (57561, 10)
